In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
ratings = pd.read_csv("ratings.csv")
# ================= USER-ITEM MATRIX =================
user_item_matrix = ratings.pivot_table(
    index='userId',
    columns='movieId',
    values='rating'
).fillna(0)
# ================= USER SIMILARITY =================
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_item_matrix.index,
    columns=user_item_matrix.index
)
# ================= RECOMMEND FUNCTION =================
def recommend_movies(user_id, top_n=5):
    if user_id not in user_item_matrix.index:
        return "User not found!"

    # Similar users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:6]

    # Weighted ratings
    weighted_ratings = np.zeros(user_item_matrix.shape[1])

    for sim_user, score in similar_users.items():
        weighted_ratings += score * user_item_matrix.loc[sim_user].values

    # Normalize
    weighted_ratings /= similar_users.sum()

    # Get unseen movies
    user_rated = user_item_matrix.loc[user_id]
    unseen_movies = user_rated[user_rated == 0].index

    scores = list(zip(unseen_movies, weighted_ratings[user_item_matrix.columns.get_indexer(unseen_movies)]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    recommended_ids = [movie for movie, _ in scores[:top_n]]

    return recommended_ids
print(recommend_movies(user_id=1))

[1200, 1610, 541, 589, 1036]


In [4]:
# MOVIELENS TOP-10 MOVIES (100k)

import pandas as pd
import urllib.request
import zipfile

# Download dataset
url = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
urllib.request.urlretrieve(url, "ml-100k.zip")

# Extract dataset
with zipfile.ZipFile("ml-100k.zip", 'r') as zip_ref:
    zip_ref.extractall()

# Load ratings
ratings = pd.read_csv("ml-100k/u.data", sep="\t",
                      names=["user_id", "movie_id", "rating", "timestamp"])

# Load movie titles
movies = pd.read_csv("ml-100k/u.item", sep="|", encoding="latin-1",
                     names=["movie_id", "title"] + [str(i) for i in range(22)])

# Merge datasets
data = pd.merge(ratings, movies, on="movie_id")

# Compute average rating and count
avg_rating = data.groupby("title")["rating"].mean()
count = data.groupby("title")["rating"].count()

# Combine both
movie_stats = pd.DataFrame({
    "average_rating": avg_rating,
    "rating_count": count
})

# Filter popular movies
movie_stats = movie_stats[movie_stats["rating_count"] > 100]

# Get top 10 movies
top10 = movie_stats.sort_values(by="average_rating", ascending=False).head(10)

# Output
print("Top 10 Movies:\n")
print(top10)

Top 10 Movies:

                                  average_rating  rating_count
title                                                         
Close Shave, A (1995)                   4.491071           112
Schindler's List (1993)                 4.466443           298
Wrong Trousers, The (1993)              4.466102           118
Casablanca (1942)                       4.456790           243
Shawshank Redemption, The (1994)        4.445230           283
Rear Window (1954)                      4.387560           209
Usual Suspects, The (1995)              4.385768           267
Star Wars (1977)                        4.358491           583
12 Angry Men (1957)                     4.344000           125
Citizen Kane (1941)                     4.292929           198
